# 10 – Free-space boxes: how they are built and what they encode

Peter's pipeline turns the 2 cm occupied / free / unknown map (step 1) into scene tokens (step 3,
`geometry/block_tokens.py`):

1. **Block labels.** The map is cut into blocks of `B`³ voxels (default 8³ = 16 cm). A block is
   **surface** if any voxel is occupied, else **free** if it has at least as many free as unknown voxels, else
   **unknown**. So a "free" block can still contain unknown voxels.
2. **Surface tokens:** one per surface block, at the mean of its occupied voxels, with the mean Qwen3-VL image feature.
3. **Box tokens:** free blocks (and unknown blocks of the chosen scope, default `"room"`) are merged into
   axis-aligned boxes by `merge_boxes`:
   * scan the grid for the first free block not yet used,
   * grow a run along the 1st growth axis, extend the run to a rectangle along the 2nd axis, extend the rectangle
     to a box along the 3rd axis – every extension needs the whole added slab to be free and unused, and each side
     is capped (default 1.5 m = 9 blocks),
   * mark it used and continue; repeat for all 6 axis orders and keep the order with the fewest boxes.

**What the model gets for a box** (`model.py`): no image feature at all, only geometry –

$$\text{box token} = \psi(\text{size}) + \phi(\text{center}) + e_\text{type}, \qquad
\text{plus 3-D RoPE at the box center (1 position = 16 cm)}$$

with $\psi, \phi$ Fourier-feature MLPs (wavelengths 32 m … 8 cm) and $e_\text{type}$ a learned free / unknown
embedding. Everything the model knows about free space is *where* the boxes are and *how big* they are.

**Prerequisite:** run Peter's steps 1 and 3 for the scene first (from the repo root):
```bash
.venv/bin/python scripts/step1_reconstruction.py --data-root data/scannetpp --scene 1ada7a0617
.venv/bin/python scripts/step3_tokens.py --scene 1ada7a0617
```

Sections: 1 load · 2 block labels · 3 box growth step by step · 4 box statistics · 5 purity and coverage vs. the
2 cm map · 6 sensitivity (growth order, cap, block size) · 7 3D view · 8 what the model sees.

In [ ]:
%matplotlib inline
from itertools import permutations
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle
import plotly.graph_objects as go

from volumetric_reasoning.geometry.block_map import BlockMap
from volumetric_reasoning.geometry.block_tokens import block_labels, merge_boxes, partition_mask, tokenize, unknown_scope
from volumetric_reasoning.geometry.labels import FREE, OCCUPIED, UNKNOWN
from volumetric_reasoning.render3d import LABEL_HEX, box_mesh, face_shade, facecolors

# ---- settings (defaults = Peter's step-3 defaults) -------------------------------------
SCENE = "1ada7a0617"
BLOCK_VOXELS = 8            # 8 x 2 cm = 16 cm blocks
CAP_M = 1.5                 # max. box side in meters (None = no cap)
UNKNOWN_SCOPE = "room"      # "all", "room" or "frontier"
Z_CUT = 2.5                 # ceiling cut for the 3D views (m)
SAVE_HTML = False           # also write the plotly views to outputs/<scene>/notebook10/
# -----------------------------------------------------------------------------------------

root = Path.cwd() if (Path.cwd() / "outputs").exists() else Path.cwd().parent
out = root / "outputs" / SCENE
bm = BlockMap.load(out / "step1_reconstruction" / "block_map.npz")
state = bm.to_dense_states()                                   # (X, Y, Z) 2 cm labels
tok, lab = tokenize(state, BLOCK_VOXELS, bm.origin, bm.h, CAP_M, UNKNOWN_SCOPE)
B, ext = BLOCK_VOXELS, tok.block_size_m
MAX_CELLS = None if CAP_M is None else max(1, int(np.floor(CAP_M / ext + 1e-9)))
is_free = tok.box_type == FREE

print(f"{SCENE}: 2 cm map {state.shape}, {ext * 100:.0f} cm block grid {lab.shape}, box cap {MAX_CELLS} blocks")
print(f"tokens: {tok.counts()}")
saved = out / "step3_tokens" / "tokens.npz"
if saved.exists() and (BLOCK_VOXELS, CAP_M, UNKNOWN_SCOPE) == (8, 1.5, "room"):
    t = np.load(saved)
    same = np.array_equal(t["box_start"], tok.box_start) and np.array_equal(t["box_stop"], tok.box_stop)
    print(f"identical to step 3's tokens.npz: {same}")

def to_world(cells):
    return bm.origin + np.asarray(cells) * ext

free_layers = np.flatnonzero((lab == FREE).any(axis=(0, 1)))
ROOM_FLOOR = float(bm.origin[2] + free_layers[0] * ext)
html_dir = out / "notebook10"

## 2. Block labels

Fine 2 cm labels vs. 16 cm block labels on horizontal slices. **Mixed** blocks (free and unknown voxels, no
surface) are decided by majority – their outline is drawn in black. Right: the free boxes cut by the same plane
(one color per box).

In [ ]:
HEIGHTS = [0.5, 1.2, 2.0]            # meters above the room floor

_, mixed = block_labels(state, B)
mixed = mixed[:lab.shape[0], :lab.shape[1], :lab.shape[2]]
fine_cmap = ListedColormap([LABEL_HEX[UNKNOWN], LABEL_HEX[FREE], LABEL_HEX[OCCUPIED]])
rng = np.random.default_rng(0)
box_colors = rng.uniform(0.15, 0.95, (len(tok.box_start), 3))
X0, Y0 = bm.origin[0], bm.origin[1]
extent_xy = [X0, X0 + lab.shape[0] * ext, Y0, Y0 + lab.shape[1] * ext]

fig, ax = plt.subplots(len(HEIGHTS), 3, figsize=(17, 5.2 * len(HEIGHTS)))
for row, hgt in zip(ax, HEIGHTS):
    z = ROOM_FLOOR + hgt
    kf = int((z - bm.origin[2]) / bm.h)
    kb = int((z - bm.origin[2]) / ext)
    row[0].imshow(state[:, :, kf].T, origin="lower", cmap=fine_cmap, vmin=0, vmax=2, extent=extent_xy, interpolation="nearest")
    row[0].set_title(f"2 cm labels, z = {hgt:.1f} m")
    row[1].imshow(lab[:, :, kb].T, origin="lower", cmap=fine_cmap, vmin=0, vmax=2, extent=extent_xy, interpolation="nearest")
    for i, j in np.argwhere(mixed[:, :, kb]):
        row[1].add_patch(Rectangle((X0 + i * ext, Y0 + j * ext), ext, ext, fill=False, ec="k", lw=0.6))
    row[1].set_title(f"{ext * 100:.0f} cm block labels (black outline = mixed, decided by majority)")
    row[2].imshow(lab[:, :, kb].T == OCCUPIED, origin="lower", cmap=ListedColormap(["white", "#eeeeee"]), extent=extent_xy)
    cut = is_free & (tok.box_start[:, 2] <= kb) & (tok.box_stop[:, 2] > kb)
    for b in np.flatnonzero(cut):
        (x0, y0), (x1, y1) = tok.box_start[b, :2], tok.box_stop[b, :2]
        row[2].add_patch(Rectangle((X0 + x0 * ext, Y0 + y0 * ext), (x1 - x0) * ext, (y1 - y0) * ext,
                                   fc=box_colors[b], ec="k", lw=0.5, alpha=0.85))
    row[2].set_title(f"{cut.sum()} free boxes cut by this plane (gray = surface blocks)")
    for a in row:
        a.set_aspect("equal"); a.set_xlim(extent_xy[:2]); a.set_ylim(extent_xy[2:])
ax[0, 0].legend(handles=[Patch(color=LABEL_HEX[s], label=n) for s, n in [(OCCUPIED, "occupied / surface"), (FREE, "free"), (UNKNOWN, "unknown")]],
                loc="upper left", fontsize=9)
plt.tight_layout()

n_free_blocks, n_mixed_free = int((lab == FREE).sum()), int((mixed & (lab == FREE)).sum())
print(f"free blocks: {n_free_blocks:,}, of which mixed (contain unknown voxels): {n_mixed_free:,} ({n_mixed_free / n_free_blocks * 100:.0f} %)")
print(f"unknown blocks: {int((lab == UNKNOWN).sum()):,} (in scope '{UNKNOWN_SCOPE}': {int(unknown_scope(lab, UNKNOWN_SCOPE).sum()):,}), "
      f"surface blocks: {int((lab == OCCUPIED).sum()):,}")

## 3. Box growth step by step

`partition_traced` is a copy of Peter's `partition_mask` that additionally records each box after every growth
step (run → rectangle → box). The cell checks that it gives exactly the same boxes as his code.

In [ ]:
def partition_traced(mask, growth_order, max_extent=None):
    # Same algorithm as block_tokens.partition_mask; returns (start, stop_after_run, stop_after_rect, stop), x/y/z order.
    limit = np.array(mask.shape, np.int64) if max_extent is None else np.asarray(max_extent, np.int64)
    mx, my, mz = (int(limit[a]) for a in growth_order)
    scan_axes = tuple(reversed(growth_order))
    available = np.ascontiguousarray(mask.transpose(scan_axes)).copy()
    nz, ny, nx = available.shape
    rec = []
    for z in range(nz):
        for y in range(ny):
            for x in np.flatnonzero(available[z, y]):
                if not available[z, y, x]:
                    continue
                blocked = np.flatnonzero(~available[z, y, x:])
                x1 = min(int(x + blocked[0]) if len(blocked) else nx, x + mx)
                y1 = y + 1
                while y1 < ny and y1 - y < my and available[z, y1, x:x1].all():
                    y1 += 1
                z1 = z + 1
                while z1 < nz and z1 - z < mz and available[z1, y:y1, x:x1].all():
                    z1 += 1
                available[z:z1, y:y1, x:x1] = False
                rec.append([(z, y, x), (z + 1, y + 1, x1), (z + 1, y1, x1), (z1, y1, x1)])
    inverse = np.argsort(scan_axes)
    rec = np.asarray(rec, np.int64).reshape(-1, 4, 3)[:, :, inverse]
    return rec[:, 0], rec[:, 1], rec[:, 2], rec[:, 3]


free_mask = lab == FREE
limit = None if MAX_CELLS is None else (MAX_CELLS,) * 3
order_counts = {}
best = None
for order in permutations(range(3)):
    traced = partition_traced(free_mask, order, limit)
    order_counts[order] = len(traced[0])
    if best is None or len(traced[0]) < len(best[1][0]):
        best = (order, traced)
ORDER, (g_start, g_run, g_rect, g_stop) = best
ref_start, ref_stop = merge_boxes(free_mask, MAX_CELLS)
assert np.array_equal(g_start, ref_start) and np.array_equal(g_stop, ref_stop), "traced boxes differ from merge_boxes"
axis = "xyz"
print("free boxes per growth order (1st, 2nd, 3rd axis):",
      {"".join(axis[a] for a in o): n for o, n in order_counts.items()})
print(f"chosen: {''.join(axis[a] for a in ORDER)} -> {len(g_start)} boxes (identical to merge_boxes)")

### a) One box: run → rectangle → box

Pick a box with `BOX` (index in creation order; `None` = the largest). Gray: free boxes created before it,
light blue: free blocks still available. Use the slider to step through the three growth steps.

In [ ]:
BOX = None
SHRINK = 0.01                      # m, gap between neighbouring boxes

vol_cells = np.prod(g_stop - g_start, axis=1)
b = int(np.argmax(vol_cells)) if BOX is None else BOX

def mesh_trace(lo, hi, rgb, name, opacity=1.0, shrink=SHRINK, legend=True):
    lo = np.asarray(lo, float).reshape(-1, 3) + shrink; hi = np.asarray(hi, float).reshape(-1, 3) - shrink
    hi = np.minimum(hi, Z_CUT * np.array([np.inf, np.inf, 1]) + [0, 0, ROOM_FLOOR])
    ok = (hi > lo).all(1)
    lo, hi, rgb = lo[ok], hi[ok], np.asarray(rgb).reshape(-1, 3)[ok] if np.ndim(rgb) == 2 else rgb
    if not len(lo):
        return go.Mesh3d(x=[], y=[], z=[], name=name, showlegend=legend)
    V, F, box, _ = box_mesh(lo, hi)
    col = (rgb[box] if np.ndim(rgb) == 2 else np.tile(rgb, (len(F), 1))) * face_shade(V, F)[:, None]
    return go.Mesh3d(x=V[:, 0], y=V[:, 1], z=V[:, 2], i=F[:, 0], j=F[:, 1], k=F[:, 2], facecolor=facecolors(col),
                     flatshading=True, opacity=opacity, name=name, showlegend=legend, hoverinfo="skip",
                     lighting=dict(ambient=1, diffuse=0, specular=0, fresnel=0))

def layout3d(title, height=720, lo=None, hi=None):
    # Fixed axis ranges (default: the whole grid up to the ceiling cut), so the view does not jump between slider steps.
    lo = to_world([0, 0, 0]) if lo is None else np.asarray(lo, float)
    hi = np.minimum(to_world(lab.shape), [np.inf, np.inf, ROOM_FLOOR + Z_CUT]) if hi is None else np.asarray(hi, float)
    span = hi - lo
    axes = {f"{a}axis": dict(range=[lo[i], hi[i]], visible=False) for i, a in enumerate("xyz")}
    return dict(title=title, height=height, margin=dict(l=0, r=0, t=50, b=0), legend=dict(x=0.01, y=0.95),
                scene=dict(aspectmode="manual", aspectratio=dict(zip("xyz", span / span.max())), **axes,
                           camera=dict(eye=dict(x=-0.9, y=-1.3, z=1.3))))

# neighbourhood of the box: everything within 4 blocks
pad = 4
nlo, nhi = np.maximum(g_start[b] - pad, 0), np.minimum(g_stop[b] + pad, lab.shape)
def near(lo, hi):
    return ((hi > nlo) & (lo < nhi)).all(1)

before = np.arange(b)[near(g_start[:b], g_stop[:b])]
used = np.zeros(lab.shape, bool)
for i in range(b):
    used[tuple(slice(s, e) for s, e in zip(g_start[i], g_stop[i]))] = True
avail = np.argwhere(free_mask & ~used)
avail = avail[near(avail, avail + 1)]

static = [mesh_trace(to_world(g_start[before]), to_world(g_stop[before]), np.array([0.7, 0.7, 0.7]),
                     f"earlier boxes ({len(before)})", opacity=0.25),
          mesh_trace(to_world(avail), to_world(avail + 1), np.array([0.62, 0.79, 0.88]),
                     f"available free blocks ({len(avail)})", opacity=0.18, shrink=0.04)]
steps = [("1. run", g_run[b]), ("2. rectangle", g_rect[b]), ("3. box", g_stop[b])]
red = np.array([0.85, 0.15, 0.15])
fig = go.Figure(static + [mesh_trace(to_world(g_start[b]), to_world(steps[0][1]), red, f"box {b}")],
                frames=[go.Frame(name=n, data=[mesh_trace(to_world(g_start[b]), to_world(s), red, f"box {b}: {n}")],
                                 traces=[2]) for n, s in steps])
size = (g_stop[b] - g_start[b]) * ext
fig.update_layout(**layout3d(f"box {b} of {len(g_start)}: grows along {axis[ORDER[0]]}, then {axis[ORDER[1]]}, then {axis[ORDER[2]]} "
                             f"-> {size[0]:.2f} x {size[1]:.2f} x {size[2]:.2f} m",
                             lo=to_world(nlo), hi=np.minimum(to_world(nhi), [np.inf, np.inf, ROOM_FLOOR + Z_CUT])),
                  sliders=[dict(active=0, currentvalue=dict(prefix="step: "), pad=dict(t=10),
                                steps=[dict(label=n, method="animate", args=[[n], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                                       for n, _ in steps])])
fig.show()
if SAVE_HTML:
    html_dir.mkdir(parents=True, exist_ok=True); fig.write_html(html_dir / "box_growth_one.html", include_plotlyjs="cdn")

### b) All boxes in creation order

The slider adds the boxes in the order `merge_boxes` creates them (color = creation order, the newest box in red).
The scan's outer loop runs along the **3rd** growth axis (for the chosen order `zxy` that is y), so the boxes sweep
through the room slice by slice along that axis – not floor to ceiling.

In [ ]:
N_STEPS = 40

n_box = len(g_start)
ks = np.unique(np.linspace(1, n_box, N_STEPS).round().astype(int))
order_rgb = plt.cm.viridis(np.linspace(0, 1, n_box))[:, :3]
lo_w, hi_w = to_world(g_start), to_world(g_stop)

def frame_data(k):
    return [mesh_trace(lo_w[:k - 1], hi_w[:k - 1], order_rgb[:k - 1], f"boxes 1–{k - 1}"),
            mesh_trace(lo_w[k - 1:k], hi_w[k - 1:k], red, f"box {k}")]

fig = go.Figure(frame_data(ks[0]), frames=[go.Frame(name=str(k), data=frame_data(k), traces=[0, 1]) for k in ks])
fig.update_layout(**layout3d(f"{n_box} free boxes in creation order (ceiling cut at {Z_CUT} m)"),
                  sliders=[dict(active=0, currentvalue=dict(prefix="boxes: "), pad=dict(t=10),
                                steps=[dict(label=str(k), method="animate",
                                            args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for k in ks])],
                  updatemenus=[dict(type="buttons", x=0.05, y=0.05, showactive=False,
                                    buttons=[dict(label="▶ play", method="animate",
                                                  args=[None, dict(frame=dict(duration=150, redraw=True), fromcurrent=True)])])])
fig.show()
if SAVE_HTML:
    html_dir.mkdir(parents=True, exist_ok=True); fig.write_html(html_dir / "box_growth_all.html", include_plotlyjs="cdn")

## 4. Box statistics (free boxes)

Sizes in blocks are integers 1 … `MAX_CELLS` per axis, so the model only ever sees a small "vocabulary" of box sizes.

In [ ]:
cells = (tok.box_stop - tok.box_start)[is_free]          # (Nf, 3) side lengths in blocks
size_m = cells * ext
vol = np.prod(size_m, axis=1)
n = len(cells)
capped = (cells == MAX_CELLS).any(1) if MAX_CELLS else np.zeros(n, bool)
single = (cells == 1).all(1)
aspect = cells.max(1) / cells.min(1)
center_h = tok.box_center[is_free, 2] - ROOM_FLOOR

print(f"{n} free boxes, total {vol.sum():.1f} m³ ({int(np.prod(cells, 1).sum()):,} blocks)")
print(f"  volume: median {np.median(vol):.3f} m³, mean {vol.mean():.3f} m³, max {vol.max():.2f} m³")
print(f"  single-block boxes ({ext * 100:.0f} cm cubes): {single.sum()} ({single.mean() * 100:.0f} % of boxes, "
      f"{vol[single].sum() / vol.sum() * 100:.1f} % of free volume)")
print(f"  boxes with a side at the cap ({CAP_M} m): {capped.sum()} ({capped.mean() * 100:.0f} %), "
      f"holding {vol[capped].sum() / vol.sum() * 100:.0f} % of the free volume")
print(f"  distinct sizes (sx, sy, sz): {len(np.unique(cells, axis=0))} of {MAX_CELLS ** 3 if MAX_CELLS else '∞'} possible")
order_v = np.sort(vol)[::-1]
cum = np.cumsum(order_v) / order_v.sum()
for frac in (0.5, 0.8, 0.9):
    k = int(np.searchsorted(cum, frac)) + 1
    print(f"  {frac * 100:.0f} % of the free volume lies in the {k} largest boxes ({k / n * 100:.0f} % of boxes)")

fig, ax = plt.subplots(1, 4, figsize=(21, 4.2))
ax[0].hist(vol, bins=np.logspace(np.log10(ext ** 3), np.log10(vol.max() * 1.01), 30), color=LABEL_HEX[FREE], ec="k", lw=0.3)
ax[0].set_xscale("log"); ax[0].set_xlabel("box volume [m³]"); ax[0].set_ylabel("boxes"); ax[0].set_title("volume per box")
bins = np.arange(1, (MAX_CELLS or cells.max()) + 2) - 0.5
for a, c in zip("xyz", cells.T):
    ax[1].hist(c, bins=bins, histtype="step", lw=2, label=a)
ax[1].set_xlabel(f"side length [blocks of {ext * 100:.0f} cm]"); ax[1].set_title("side lengths per axis"); ax[1].legend()
ax[2].plot(np.arange(1, n + 1) / n * 100, cum * 100)
ax[2].set_xlabel("% of boxes (largest first)"); ax[2].set_ylabel("% of free volume"); ax[2].grid(alpha=0.3)
ax[2].set_title("how concentrated is the free volume?")
ax[3].scatter(center_h, vol, s=8, c=np.log(aspect + 1e-9), cmap="plasma")
ax[3].set_yscale("log"); ax[3].set_xlabel("box center above floor [m]"); ax[3].set_ylabel("volume [m³]")
ax[3].set_title("volume vs. height (color = log aspect ratio)")
plt.tight_layout()

## 5. Purity and coverage vs. the 2 cm map

* **Purity:** how much of a free box is actually free at 2 cm? (Mixed blocks bring unknown voxels into free boxes;
  occupied voxels cannot occur, since any occupied voxel makes the block a surface block.)
* **Coverage:** where does the 2 cm free space end up? Inside free boxes, inside surface blocks (free space next to
  surfaces is not encoded as free – only implicitly by the surface token), or in unknown blocks.

In [ ]:
def box_purity(t, state):
    # (Nf, 3) fraction of unknown / free / occupied 2 cm voxels in each free box.
    out = []
    Bv = int(round(t.block_size_m / bm.h))
    for s, e in zip(t.box_start[t.box_type == FREE], t.box_stop[t.box_type == FREE]):
        sub = state[tuple(slice(a * Bv, b * Bv) for a, b in zip(s, e))]
        out.append(np.bincount(sub.ravel(), minlength=3)[:3] / sub.size)
    return np.array(out)

def free_coverage(lab, Bv):
    # Where the 2 cm free voxels lie, by the label of their block.
    up = lab.repeat(Bv, 0).repeat(Bv, 1).repeat(Bv, 2)[:state.shape[0], :state.shape[1], :state.shape[2]]
    f = state == FREE
    return {name: float((f & (up == s)).sum() / f.sum()) for s, name in [(FREE, "free blocks"), (OCCUPIED, "surface blocks"), (UNKNOWN, "unknown blocks")]}

pur = box_purity(tok, state)
w = vol / vol.sum()
print(f"free boxes, volume-weighted: {pur[:, FREE] @ w * 100:.1f} % free, {pur[:, UNKNOWN] @ w * 100:.1f} % unknown, "
      f"{pur[:, OCCUPIED] @ w * 100:.1f} % occupied voxels")
print(f"boxes that are < 90 % free: {(pur[:, FREE] < 0.9).sum()} of {len(pur)}")
cov = free_coverage(lab, B)
print("2 cm free voxels by block label: " + ", ".join(f"{k} {v * 100:.1f} %" for k, v in cov.items()))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(pur[:, FREE] * 100, bins=40, color=LABEL_HEX[FREE], ec="k", lw=0.3)
ax[0].set_xlabel("% of the box's 2 cm voxels that are free"); ax[0].set_ylabel("boxes"); ax[0].set_title("purity of free boxes")
ax[1].barh(list(cov), [v * 100 for v in cov.values()], color=[LABEL_HEX[FREE], LABEL_HEX[OCCUPIED], LABEL_HEX[UNKNOWN]])
ax[1].set_xlabel("% of all 2 cm free voxels"); ax[1].set_title("where the measured free space ends up")
plt.tight_layout()

## 6. Sensitivity: growth order, size cap, block size

Token count of the free boxes, their median volume and how much of the 2 cm free space they cover.

In [ ]:
CAPS = [None, 3.0, 2.0, 1.5, 1.0, 0.5]
BLOCKS = [4, 8, 16]                       # voxels per block side: 8 / 16 / 32 cm

rows = []
for Bv in BLOCKS:
    for cap in CAPS:
        t, l = tokenize(state, Bv, bm.origin, bm.h, cap, UNKNOWN_SCOPE)
        f = t.box_type == FREE
        v = np.prod(t.box_size[f], 1)
        rows.append(dict(block_cm=round(Bv * bm.h * 100), cap="none" if cap is None else f"{cap:g} m", free_boxes=int(f.sum()),
                         unknown_boxes=int((~f).sum()), surface=len(t.surface_position),
                         median_m3=float(np.median(v)), coverage=free_coverage(l, Bv)["free blocks"]))
print(f"{'block':>6} {'cap':>6} | {'free':>6} {'unk':>5} {'surface':>8} | {'median vol':>10} {'free covered':>13}")
for r in rows:
    print(f"{r['block_cm']:>4}cm {r['cap']:>6} | {r['free_boxes']:>6} {r['unknown_boxes']:>5} {r['surface']:>8} | "
          f"{r['median_m3']:>8.3f}m³ {r['coverage'] * 100:>12.1f}%")

fig, ax = plt.subplots(1, 3, figsize=(19, 4.2))
ax[0].bar(["".join(axis[a] for a in o) for o in order_counts], list(order_counts.values()), color="#8da0cb")
ax[0].axhline(min(order_counts.values()), color="k", lw=0.8, ls="--")
ax[0].set_ylabel("free boxes"); ax[0].set_title(f"growth order ({ext * 100:.0f} cm, cap {CAP_M} m); min is used")
for Bv in BLOCKS:
    rs = [r for r in rows if r["block_cm"] == round(Bv * bm.h * 100)]
    ax[1].plot([r["cap"] for r in rs], [r["free_boxes"] for r in rs], "o-", label=f"{rs[0]['block_cm']} cm blocks")
    ax[2].scatter([r["free_boxes"] for r in rs], [r["coverage"] * 100 for r in rs], label=f"{rs[0]['block_cm']} cm blocks")
ax[1].set_yscale("log"); ax[1].set_xlabel("box size cap"); ax[1].set_ylabel("free boxes"); ax[1].legend(); ax[1].set_title("free tokens vs. cap")
ax[2].set_xscale("log"); ax[2].set_xlabel("free boxes (log)"); ax[2].set_ylabel("% of 2 cm free space in free boxes")
ax[2].set_title("coverage is set by the block size; the cap only changes the box count"); ax[2].legend()
plt.tight_layout()

## 7. 3D view

Free boxes (one random color each, slightly shrunk to show the partition), unknown boxes (gray, hidden by default –
click the legend), surface tokens as small dots. Ceiling cut at `Z_CUT`.

In [ ]:
SHOW_UNKNOWN = False

lo_all, hi_all = to_world(tok.box_start), to_world(tok.box_stop)
sp = tok.surface_position
sp = sp[sp[:, 2] < ROOM_FLOOR + Z_CUT]
traces = [mesh_trace(lo_all[is_free], hi_all[is_free], box_colors[is_free], f"free boxes ({is_free.sum()})", shrink=0.02),
          mesh_trace(lo_all[~is_free], hi_all[~is_free], np.array([0.6, 0.6, 0.6]), f"unknown boxes ({(~is_free).sum()})",
                     opacity=0.5, shrink=0.02),
          go.Scatter3d(x=sp[:, 0], y=sp[:, 1], z=sp[:, 2], mode="markers", name=f"surface tokens ({len(tok.surface_position):,})",
                       marker=dict(size=1.5, color=LABEL_HEX[OCCUPIED]))]
traces[1].visible = True if SHOW_UNKNOWN else "legendonly"
fig = go.Figure(traces)
fig.update_layout(**layout3d(f"{SCENE}: {ext * 100:.0f} cm blocks, cap {CAP_M} m, unknown scope '{UNKNOWN_SCOPE}'", height=780))
fig.show()
if SAVE_HTML:
    html_dir.mkdir(parents=True, exist_ok=True); fig.write_html(html_dir / "boxes_3d.html", include_plotlyjs="cdn")

## 8. What the model sees for a free box

Each box token is built from exactly these numbers (plus its type). The decoder gets their Fourier encodings and
the RoPE position of the center – nothing from the images.

In [ ]:
idx = np.argsort(-vol)[:5].tolist() + np.flatnonzero(single)[:3].tolist()
fb = np.flatnonzero(is_free)
print(f"{'box':>5} {'center x, y, z [m]':>24} {'size x, y, z [m]':>22} {'RoPE (t, h, w) offset':>24}")
all_pos = np.concatenate([tok.surface_position, tok.box_center])
rel = (tok.box_center - all_pos.min(0)) / 0.16               # model.ROPE_UNIT_M
for i in idx:
    c, s, r = tok.box_center[fb[i]], tok.box_size[fb[i]], rel[fb[i]][[2, 1, 0]]
    print(f"{i:>5} {c[0]:8.2f}{c[1]:8.2f}{c[2]:8.2f} {s[0]:7.2f}{s[1]:7.2f}{s[2]:7.2f} {r[0]:8.1f}{r[1]:8.1f}{r[2]:8.1f}")
print(f"\nThe free space of this scene = {is_free.sum()} such tuples; the surface = {len(tok.surface_position):,} tokens "
      f"with a {ext * 100:.0f} cm-block image feature each.")

## Findings

First run, 1ada7a0617, Peter's defaults (32 frames, 16 cm blocks, cap 1.5 m, unknown scope "room"):

- **509 free boxes** encode 8,868 free blocks (36 m³), ~17 blocks per token. 5 % of free blocks are mixed
  (contain unknown voxels); the boxes are still 99.3 % free at 2 cm (volume-weighted), 62 boxes are < 90 % free.
- **Very uneven sizes:** 50 % of the free volume lies in the 41 largest boxes (8 %); 23 % of boxes hit the 1.5 m
  cap and hold 57 % of the volume; 11 % are single 16 cm cubes (0.6 % of the volume) – fragments next to
  furniture and walls. Only 137 distinct box sizes occur.
- **Growth order matters:** 509 (zxy, chosen) to 590 (yxz) boxes, i.e. up to 16 % more tokens for the same space.
- **20 % of the measured free space is not in any free box** – it lies in surface blocks (free space within 16 cm of
  a surface is only implicit in the surface token). This is set by the block size alone (8 cm: 91 % covered,
  32 cm: 61 %); the cap only changes how many boxes the same free blocks are cut into.
- A box token is just (center, size, type) – no image information. Open: does the model use box *size* at all
  (ablation: replace sizes by a constant), and would a finer free-space resolution near surfaces help?